# RBI Data Quality Pipeline: Stage 4 (Rupee and Dollar Consistency)

The reserves table holds the same money in rupees and in US dollars. Dividing
one by the other gives the implied exchange rate, which should change only a
little from week to week. This notebook checks that, and tests whether the
sudden-jump check from stage 3 works on that rate.

In [1]:
import pandas as pd
data = pd.read_csv("table2_clean.csv",parse_dates=["week_ended"])
print(data.shape)

(1334, 11)


## The implied exchange rate

For each week I divided total reserves in rupees (crores) times 10 by total
reserves in dollars (millions), which gives rupees per dollar. It runs from
about 47 in 2001 to about 95 in 2026. The last value, 94.94, is close to the
94.88 shown on the RBI portal when I first opened it.

In [2]:
rate = data["total_reserves_inr"]*10/data["total_reserves_usd"]
print(rate.head())
print(rate.tail())

0    46.631685
1    46.874678
2    46.837367
3    46.846017
4    46.831845
dtype: float64
1329    95.835938
1330    96.284462
1331    96.723855
1332    94.951530
1333    94.943076
dtype: float64


The jump check is the same function as in stage 3, copied here for now.

In [3]:
def find_jumps(values,dates,cutoff=3.5):
    changes = values.pct_change()*100
    changes = changes.dropna()
    median_change = changes.median()
    mad = (changes-median_change).abs().median()
    if mad == 0:
        return None
    scores = 0.6745 * (changes-median_change)/mad
    result = pd.DataFrame({"date":dates,"change":changes,"score":scores})
    result = result[result["score"].abs()>cutoff]
    return result

## Running the jump check on the rate

With the default cutoff of 3.5, the check flags 58 of 1,333 weeks.

In [4]:
rate_flags = find_jumps(rate,data["week_ended"])
print(len(rate_flags))
print(rate_flags)

58
           date    change     score
156  2004-04-02 -2.222008 -4.504696
191  2004-12-03 -1.746391 -3.546090
245  2005-12-16 -1.756345 -3.566152
316  2007-04-27 -2.115687 -4.290405
332  2007-08-17  2.145990  4.298999
371  2008-05-16  2.956148  5.931870
388  2008-09-12  3.061620  6.144449
392  2008-10-10  3.802367  7.637423
394  2008-10-24  2.518201  5.049188
396  2008-11-07 -2.924316 -5.920197
397  2008-11-14  3.436835  6.900694
401  2008-12-12 -1.910439 -3.876729
402  2008-12-19 -3.241680 -6.559842
404  2009-01-02  1.929897  3.863463
411  2009-02-20  2.236175  4.480766
412  2009-02-27  1.825854  3.653765
415  2009-03-20 -2.848062 -5.766507
422  2009-05-08 -1.862024 -3.779148
424  2009-05-22 -4.593698 -9.284834
444  2009-10-09 -2.736983 -5.542626
474  2010-05-07  2.393069  4.796986
476  2010-05-21  3.827703  7.688488
495  2010-10-01 -2.078866 -4.216192
545  2011-09-16  2.141513  4.289975
546  2011-09-23  4.222687  8.484578
550  2011-10-21  1.855318  3.713149
551  2011-10-28 -2.273988

That is far more than the 19 flagged on dollar reserves. The flagged weeks
cluster in late 2008 to 2009, 2010 and 2011 to 2013. The week of 2010-05-21
scores 7.7 with a change of +3.8%, which is the exchange-rate move I found by
hand in stage 3. The largest weekly change anywhere is -4.6% (2009-05-22), so
no week looks like a unit mix-up between the two currencies, which would show
up as a jump of several hundred percent.

## Does the cutoff matter?

I counted the flagged weeks at different cutoffs.

In [5]:
for c in [3.5, 5, 7, 10, 15]:
    print(c, len(find_jumps(rate, data["week_ended"], c)))

3.5 58
5 18
7 5
10 0
15 0


The count falls from 58 at a cutoff of 3.5 to 18 at 5, 5 at 7, and none at 10.
The rate is a calm series, so a cutoff of 3.5 flags ordinary currency swings,
which are not errors. A fault like a unit mix-up would score far above any of
these. I have not tested that yet, since this data contains no such fault.
The benchmark stage will test it by injecting faults and measuring which
cutoff catches them.